# PPR10K Setup — 解压版 (for ECCV rebuttal)

**前提**: 你已经把 PPR10K 的官方 [Drive 文件夹 1kB2OSA...](https://drive.google.com/drive/folders/1kB2OSAGy8uc0xUXaMKoPB0HMSc-rkrLW) 加 shortcut 到 `MyDrive/datasets/`。所以 Colab 看到的路径是:
```
/content/drive/MyDrive/datasets/PPR0K_all_files_11161_zip/train_val_images_tif_360p/
    source.zip          12 GB   ← 输入(含 5x 增强,我们只用无增强版)
    target_a.zip         4.6 GB  ← expert a 输出
    target_b.zip         4.5 GB
    target_c.zip         4.7 GB
```

**策略**: 把 source + target_a (~17GB) 解压到 Colab `/content/ppr10k/`,**不要解到 Drive**(Drive 写小文件慢、训练读小文件也慢)。每次 Colab 会话重启后重解一次,大概 3-5 分钟,远比训练时读 Drive 快。

**输出**: 训练数据放 `/content/ppr10k/paired_a/{train,val}/{input,gt}/`,checkpoints 仍存到 Drive 持久化。

In [ ]:
# === Cell 1: mount + 验证 shortcut 在哪 ===
from google.colab import drive
drive.mount('/content/drive')

import os
PPR_360P = '/content/drive/MyDrive/datasets/PPR0K_all_files_11161_zip/train_val_images_tif_360p'
assert os.path.exists(PPR_360P), f"❌ {PPR_360P} 不存在 — 检查 shortcut 是否加在了 MyDrive/datasets/ 下"

print("✅ 找到 PPR10K 360p 目录")
for f in sorted(os.listdir(PPR_360P)):
    full = f'{PPR_360P}/{f}'
    if os.path.isfile(full):
        sz_gb = os.path.getsize(full) / 1e9
        print(f"  📄 {f} ({sz_gb:.2f} GB)")
    else:
        print(f"  📂 {f}/")

In [ ]:
# === Cell 2: 解压 source.zip 和 target_a.zip 到 /content/ ===
# Colab /content 通常 100GB+ 可用,17GB 没压力。每次 session 重启要重跑这个 cell
EXPERT = 'a'  # 选 a / b / c — 论文一般用 a
WORK = '/content/ppr10k_raw'

import os, time
os.makedirs(WORK, exist_ok=True)

for zip_name in ['source.zip', f'target_{EXPERT}.zip']:
    src = f'{PPR_360P}/{zip_name}'
    target_dir = zip_name.replace('.zip', '')
    out_dir = f'{WORK}/{target_dir}'
    
    if os.path.exists(out_dir) and len(os.listdir(out_dir)) > 100:
        print(f"⏭️  {zip_name} 已经解过了 ({len(os.listdir(out_dir))} 文件),跳过")
        continue
    
    print(f"📦 解压 {zip_name} ({os.path.getsize(src)/1e9:.2f} GB) ...")
    t0 = time.time()
    !unzip -q -o {src} -d {WORK}/
    dt = time.time() - t0
    print(f"   ✅ 用时 {dt:.0f}s")

# 看解压后的结构
print("\n=== /content/ppr10k_raw/ 结构 ===")
!ls -la {WORK}/
for d in sorted(os.listdir(WORK)):
    full = f'{WORK}/{d}'
    if os.path.isdir(full):
        files = sorted(os.listdir(full))
        print(f"\n📂 {d}/ ({len(files)} 个文件)")
        print(f"   头 5: {files[:5]}")
        print(f"   尾 3: {files[-3:]}")

In [ ]:
# === Cell 3: 整理成 paired 结构 (用 symlink 不复制) ===
# PPR10K 命名约定: source 含增强 (0_0.tif 是无增强,0_1.tif ... 0_4.tif 是增强)
# target 通常无增强后缀 (0.tif)。我们只用 _0.tif 配对。
import os, re

SOURCE_DIR = f'{WORK}/source'
TARGET_DIR = f'{WORK}/target_{EXPERT}'
DEST = f'/content/ppr10k_paired_{EXPERT}'

for sub in ['train/input', 'train/gt', 'val/input', 'val/gt']:
    os.makedirs(f'{DEST}/{sub}', exist_ok=True)

src_files = sorted(os.listdir(SOURCE_DIR))
tgt_files = set(os.listdir(TARGET_DIR))

# 1. 找无增强的 source: 命名 "<id>_0.tif"
unaug_pattern = re.compile(r'^(\d+)_0\.tif$')
unaug_sources = [(int(m.group(1)), f) for f in src_files if (m := unaug_pattern.match(f))]
unaug_sources.sort()  # 按 id 数字排序
print(f"无增强 source 找到 {len(unaug_sources)} 个 (期望 11161)")

# 2. 配对 target (target 命名通常 "<id>.tif" 无后缀)
paired = []
for img_id, sf in unaug_sources:
    # 试几种 target 命名
    candidates = [f'{img_id}.tif', f'{img_id}_0.tif', sf]
    for c in candidates:
        if c in tgt_files:
            paired.append((img_id, sf, c))
            break
print(f"成功配对: {len(paired)}")
if len(paired) < 100:
    print("⚠️  配对数太少,看下面 target 文件名样本,改 candidates 列表:")
    print("   target 样本:", sorted(list(tgt_files))[:10])
    raise SystemExit

# 3. 标准 PPR10K split: id 0-8874 训练 / 8875+ 验证 (按 id 数字)
train_pairs = [(s,t) for i,s,t in paired if i < 8875]
val_pairs = [(s,t) for i,s,t in paired if i >= 8875]
print(f"Split: train={len(train_pairs)}, val={len(val_pairs)}")

def link(src, dst):
    if not os.path.exists(dst):
        try: os.symlink(src, dst)
        except FileExistsError: pass

for sf, tf in train_pairs:
    link(f'{SOURCE_DIR}/{sf}', f'{DEST}/train/input/{sf}')
    link(f'{TARGET_DIR}/{tf}', f'{DEST}/train/gt/{sf}')  # gt 用 source 文件名,paired_folder 才能匹配
for sf, tf in val_pairs:
    link(f'{SOURCE_DIR}/{sf}', f'{DEST}/val/input/{sf}')
    link(f'{TARGET_DIR}/{tf}', f'{DEST}/val/gt/{sf}')

print(f"\n✅ {DEST} 准备好")
for sub in ['train/input', 'train/gt', 'val/input', 'val/gt']:
    print(f"   {sub}: {len(os.listdir(f'{DEST}/{sub}'))} 文件")

In [ ]:
# === Cell 4: 验证 paired_folder 能加载,且 input != gt ===
import sys
sys.path.insert(0, '/content/drive/MyDrive/LoR-LUT')
from data.paired_folder import PairedFolderDataset

ds = PairedFolderDataset(
    root=DEST,
    split='train',
    in_dir='input',
    gt_dir='gt',
    exts=('.tif', '.tiff'),
    patch=0,
    augment=False
)
print(f"✅ {len(ds)} train pairs")
for i in [0, 1, 2, len(ds)//2, len(ds)-1]:
    s = ds[i]
    diff = (s['img_in'] - s['img_gt']).abs().mean().item()
    status = '✅' if diff > 0.01 else '⚠️ input==gt!'
    print(f"  [{i:5d}] {s['name']:20s} input-gt MAE={diff:.4f} {status}")

In [ ]:
# === Cell 5: 训练命令 (Colab /content 数据 + Drive ckpt) ===
import datetime
EXP_NAME = f"ppr10k_{EXPERT}_K0_R8_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}"

%cd /content/drive/MyDrive/LoR-LUT

!python train.py \
    --cfg config/default.yaml \
    --data.root /content/ppr10k_paired_{EXPERT} \
    --work_dir /content/drive/MyDrive/LoR-LUT/runs/{EXP_NAME}

## 故障排除

**Cell 2 解压超慢/卡住** —— 检查 Colab 磁盘剩余 (`!df -h /content`)。如果 < 30GB 空间,先 `!rm -rf /content/sample_data` 清出空间。

**Cell 3 配对数 < 100** —— PPR10K 命名跟我猜的不一样。运行下面看实际命名,然后改 Cell 3 的 candidates 列表:
```python
import os
print("source 样本:", sorted(os.listdir(f'{WORK}/source'))[:10])
print("target 样本:", sorted(os.listdir(f'{WORK}/target_a'))[:10])
```

**Cell 4 input==gt** —— 配对错位。Cell 3 里 `link(... target_dir/tf, .../gt/sf)` 这一行的 `tf` (target 文件名) 必须跟 `sf` (source) 是同一张图的不同版本,不是同名两份。

**训练 OOM** —— 改小 batch (`config/default.yaml` 里 `train.batch`),或者 patch (`train.patch`,PPR10K 360p 图本身不大,patch=256 应该够)。